# Screenshot → UI target → coordinates
## OmniParser v2 · Google Colab walkthrough

Upload an app screenshot, enter a target such as **play button**, **share icon**, or **settings**, and inspect the detected rectangle and its coordinates. No training, API key, browser connection, or clicking is required.

**Pipeline:** screenshot → OmniParser region detector → OCR + Florence-2 icon descriptions → target matching → ranked candidates → annotated PNG + JSON.

### Start here
1. Open **Runtime → Change runtime type**. Select **T4 GPU** and **Runtime Version: 2026.07** (Python 3.12). If the version dropdown is unavailable, connect first, open the command palette, and use **Change runtime version**. Select the compatible runtime again in each new session.
2. Run **Step 1**. If prompted, restart the session, then continue from **Step 2**.
3. Run Steps **2–4**, upload your screenshot in **Step 5**, and run **Step 6** with your target name.
4. Download the results in **Step 7**. For another target on the same screenshot, rerun only Step 6.

Coordinates use the full uploaded image as displayed, after EXIF orientation correction: `[x1, y1, x2, y2]`, origin at the **top-left**, x increases right, y increases down. The center is a reference point inside that rectangle; it does not establish that an element is clickable.

**Limits:** invisible or hover-hidden controls cannot be recovered from a still screenshot. A pause icon is not a visible play icon. Several similar controls produce an `ambiguous` result until you narrow the search region. This is a prototype; test real screenshots before relying on it.


## Step 1 · Install the screenshot-only dependencies

Start with a fresh Colab GPU session using **runtime version 2026.07 (Python 3.12)**. Colab's latest runtime uses Python 3.13; the older runtime avoids incompatible wheels for these pinned dependencies. See [Google's runtime version instructions](https://research.google.com/colaboratory/runtime-version-faq.html). The notebook uses the pretrained OmniParser weights directly, avoiding the repository's unrelated browser-agent dependencies and eager initialization of two OCR engines.

**EasyOCR** is the default. To use **PaddleOCR**, change the dropdown before running this cell; both implement the same output format. PaddleOCR runs on CPU to avoid a second GPU runtime competing with PyTorch. The pins below target Python **3.10–3.12** and the Florence-2 remote-code API. Colab's existing PyTorch/CUDA installation is retained.


In [ ]:
# @title Install dependencies (run once per fresh runtime)
OCR_BACKEND = "easyocr" # @param ["easyocr", "paddleocr"]

import sys
import subprocess
from importlib import metadata

if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError("Use Runtime → Change runtime type → Runtime Version 2026.07 (Python 3.12), then rerun Step 1.")

loaded_before_install = {
    name: getattr(sys.modules[name], "__version__", None)
    for name in ("numpy", "transformers", "cv2", "PIL", "tokenizers", "huggingface_hub")
    if name in sys.modules
}
packages = [
    "numpy==1.26.4", "Pillow==11.1.0", "transformers==4.49.0",
    "huggingface_hub==0.30.2", "accelerate==1.6.0", "timm==1.0.15",
    "einops==0.8.0", "ultralytics==8.3.203", "scikit-image==0.24.0",
]
if OCR_BACKEND == "paddleocr":
    packages += ["paddlepaddle==2.6.2", "paddleocr==2.9.1",
                 "opencv-python==4.10.0.84", "opencv-contrib-python==4.10.0.84",
                 "opencv-python-headless==4.10.0.84"]
else:
    packages += ["easyocr==1.7.2", "opencv-python==4.10.0.84",
                 "opencv-python-headless==4.10.0.84"]

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)
from pathlib import Path
Path("/content/ui_ocr_backend.txt").write_text(OCR_BACKEND)

# Compare loaded modules with installed versions: do not continue with stale NumPy binaries.
dist_names = {"cv2": "opencv-python", "PIL": "Pillow"}
stale = []
for name, old_version in loaded_before_install.items():
    new_version = metadata.version(dist_names.get(name, name))
    if old_version and old_version != new_version:
        stale.append(name)
if stale:
    raise RuntimeError(
        "Installation finished. Use Runtime → Restart session, then run from Step 2. "
        "These loaded modules changed: " + ", ".join(stale)
    )
print("Installation finished. Continue with Step 2. OCR:", OCR_BACKEND)


## Step 2 · Download and load the pretrained models

The first run downloads the detector and the fine-tuned Florence-2 captioner, plus OCR weights when first used. This can take several minutes. The model and remote-code revisions are pinned and included in the result metadata.

The notebook uses the original OmniParser v2 `icon_detect/model.pt` and `icon_caption` weights. It does not substitute the general Florence model's weights for the fine-tuned icon captioner. The original detector carries its upstream AGPL-3.0 license; the caption weights are MIT licensed. See the [official model card](https://huggingface.co/microsoft/OmniParser-v2.0).


In [ ]:
import os
import gc
import re
import io
import math
import json
import hashlib
import warnings
from pathlib import Path
from importlib import metadata
from datetime import datetime, timezone
from contextlib import contextmanager
from unittest.mock import patch

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# The captioner is PyTorch-only; do not import Colab's TensorFlow or JAX stacks.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
import numpy as np
import torch
import cv2
from PIL import Image, ImageOps, ImageDraw, ImageFont, UnidentifiedImageError
from IPython.display import display
from huggingface_hub import snapshot_download
from transformers import AutoProcessor, AutoModelForCausalLM
from ultralytics import YOLO

WORK_DIR = Path("/content/screenshot_ui_detector")
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_REPO = "microsoft/OmniParser-v2.0"
MODEL_REVISION = "f55d0750e5b94db2125ef0b45b0fa4a85ddc59b4"
FLORENCE_REPO = "microsoft/Florence-2-base-ft"
FLORENCE_CODE_REVISION = "3afa95f29c831f1ea138d5d02e181fac2463e1e9"
OCR_BACKEND = Path("/content/ui_ocr_backend.txt").read_text().strip()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
if DEVICE == "cpu":
    warnings.warn("No CUDA GPU detected. CPU inference is supported but much slower; select T4 for normal use.")
else:
    print("GPU:", torch.cuda.get_device_name(0))

weights = Path(snapshot_download(
    repo_id=MODEL_REPO, revision=MODEL_REVISION,
    local_dir=str(WORK_DIR / "weights"),
    allow_patterns=["icon_detect/model.pt", "icon_detect/LICENSE",
                    "icon_caption/config.json", "icon_caption/generation_config.json",
                    "icon_caption/model.safetensors", "icon_caption/LICENSE"],
))
for relative in ("icon_detect/model.pt", "icon_caption/config.json", "icon_caption/model.safetensors"):
    if not (weights / relative).is_file():
        raise FileNotFoundError(f"Missing model file: {relative}. Rerun this cell to resume the download.")

@contextmanager
def optional_flash_attention_import():
    # Florence's dependency scanner can require flash_attn even for eager attention.
    # Ignore only that optional dependency while loading; attention remains eager.
    import transformers.dynamic_module_utils as dynamic_utils
    original = dynamic_utils.get_imports
    def get_imports(filename):
        return [name for name in original(filename) if name != "flash_attn"]
    with patch.object(dynamic_utils, "get_imports", get_imports):
        yield

detector = YOLO(str(weights / "icon_detect/model.pt"))
with optional_flash_attention_import():
    caption_processor = AutoProcessor.from_pretrained(
        FLORENCE_REPO, revision=FLORENCE_CODE_REVISION, trust_remote_code=True,
    )
    caption_model = AutoModelForCausalLM.from_pretrained(
        str(weights / "icon_caption"),
        code_revision=FLORENCE_CODE_REVISION,
        trust_remote_code=True, torch_dtype=DTYPE, attn_implementation="eager",
    ).to(DEVICE).eval()

OCR_ENGINE = None  # Loaded once, on demand.
PARSE_CACHE = {}
print("Models ready. OCR backend:", OCR_BACKEND, "| Device:", DEVICE)


## Step 3 · Screenshot parsing and coordinate helpers

The detector proposes UI regions; OCR supplies visible labels, and Florence-2 describes unlabeled icons. Original image dimensions are preserved. Empty OCR results are valid and do not stop icon detection. Confidence from the region detector or OCR is kept separately from target matching.


In [ ]:
def clamp_box(box, width, height):
    """Pixel xyxy with exclusive right/bottom edges. Return None for invalid boxes."""
    values = [float(v) for v in box]
    if len(values) != 4 or not all(math.isfinite(v) for v in values):
        return None
    x1 = max(0, min(width, math.floor(values[0])))
    y1 = max(0, min(height, math.floor(values[1])))
    x2 = max(0, min(width, math.ceil(values[2])))
    y2 = max(0, min(height, math.ceil(values[3])))
    return [x1, y1, x2, y2] if x2 > x1 and y2 > y1 else None

def area(box):
    return max(0, box[2] - box[0]) * max(0, box[3] - box[1])

def intersection(a, b):
    return max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))

def iou(a, b):
    overlap = intersection(a, b)
    return overlap / max(1, area(a) + area(b) - overlap)

def coordinate_fields(box, width, height):
    x1, y1, x2, y2 = box
    return {
        "box": list(box),
        "center": [min(width - 1, (x1 + x2) // 2), min(height - 1, (y1 + y2) // 2)],
        "box_normalized": [round(x1 / width, 6), round(y1 / height, 6),
                           round(x2 / width, 6), round(y2 / height, 6)],
    }

def load_screenshot(path):
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
        image.load()
    return image

def get_ocr_engine():
    global OCR_ENGINE
    if OCR_ENGINE is None:
        if OCR_BACKEND == "paddleocr":
            from paddleocr import PaddleOCR
            OCR_ENGINE = PaddleOCR(lang="en", use_angle_cls=False, use_gpu=False, show_log=False)
        else:
            import easyocr
            # CPU OCR leaves GPU memory for the icon captioner.
            OCR_ENGINE = easyocr.Reader(["en"], gpu=False)
    return OCR_ENGINE

def extract_text(image, min_confidence=0.5):
    engine = get_ocr_engine()
    rgb = np.asarray(image)
    if OCR_BACKEND == "paddleocr":
        # PaddleOCR 2.x consumes BGR and can return None or [None] on a blank image.
        pages = engine.ocr(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), cls=False) or []
        items = pages[0] if pages and pages[0] else []
        rows = [(item[0], item[1][0], item[1][1]) for item in items]
    else:
        rows = engine.readtext(rgb, detail=1, paragraph=False)
    output = []
    for polygon, text, confidence in rows:
        if float(confidence) < min_confidence or not str(text).strip():
            continue
        points = np.asarray(polygon, dtype=float)
        box = clamp_box([points[:, 0].min(), points[:, 1].min(),
                         points[:, 0].max(), points[:, 1].max()], *image.size)
        if box:
            output.append({"box": box, "text": str(text).strip(),
                           "ocr_confidence": round(float(confidence), 6)})
    return output

def caption_regions(image, boxes, batch_size=8):
    """Describe only valid icon crops. Preserve a one-to-one crop/caption mapping."""
    output = []
    for start in range(0, len(boxes), batch_size):
        crops = [image.crop(tuple(box)).resize((64, 64)) for box in boxes[start:start + batch_size]]
        inputs = caption_processor(text=["<CAPTION>"] * len(crops), images=crops,
                                   return_tensors="pt", do_resize=False)
        inputs = {key: value.to(DEVICE, dtype=DTYPE) if value.is_floating_point()
                  else value.to(DEVICE) for key, value in inputs.items()}
        with torch.inference_mode():
            generated = caption_model.generate(
                input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"],
                max_new_tokens=24, num_beams=1, do_sample=False,
            )
        captions = caption_processor.batch_decode(generated, skip_special_tokens=True)
        if len(captions) != len(crops):
            raise RuntimeError("Caption count mismatch; coordinates were not assigned to uncertain captions.")
        output.extend(text.strip() for text in captions)
    return output

def parse_screenshot(image_path, detection_threshold=0.10, ocr_threshold=0.50,
                     image_size=1280, batch_size=8):
    if not 0 < detection_threshold <= 1 or not 0 <= ocr_threshold <= 1:
        raise ValueError("Detection and OCR thresholds must be between 0 and 1.")
    if batch_size < 1 or image_size < 32:
        raise ValueError("Batch size must be positive; detector image size must be at least 32.")
    image = load_screenshot(image_path)
    width, height = image.size
    digest = hashlib.sha256(image.tobytes() + str(image.size).encode()).hexdigest()
    key = (digest, detection_threshold, ocr_threshold, image_size, batch_size, OCR_BACKEND)
    if key in PARSE_CACHE:
        return image, PARSE_CACHE[key]
    text_items = extract_text(image, ocr_threshold)
    prediction = detector.predict(source=image, conf=detection_threshold, iou=0.5,
                                  imgsz=image_size, device=DEVICE, max_det=300, verbose=False)[0]
    elements = []
    used_text = set()
    pending_captions = []
    if prediction.boxes is not None:
        boxes = prediction.boxes.xyxy.detach().cpu().tolist()
        confidences = prediction.boxes.conf.detach().cpu().tolist()
        for raw_box, confidence in zip(boxes, confidences):
            box = clamp_box(raw_box, width, height)
            if not box or min(box[2] - box[0], box[3] - box[1]) < 3:
                continue
            contained = [i for i, item in enumerate(text_items)
                         if intersection(box, item["box"]) / max(1, area(item["box"])) >= 0.8]
            contained.sort(key=lambda i: (text_items[i]["box"][1], text_items[i]["box"][0]))
            label = " ".join(text_items[i]["text"] for i in contained)
            used_text.update(contained)
            element = {
                "id": len(elements), "type": "icon", "source": "omniparser_v2",
                "description": label, "ocr_text": label,
                "detection_confidence": round(float(confidence), 6),
                "ocr_confidence": (round(min(text_items[i]["ocr_confidence"] for i in contained), 6)
                                   if contained else None),
                **coordinate_fields(box, width, height),
            }
            elements.append(element)
            if not label:
                pending_captions.append(element)
    captions = caption_regions(image, [item["box"] for item in pending_captions], batch_size)
    for element, caption in zip(pending_captions, captions):
        element["description"] = caption
    for index, item in enumerate(text_items):
        if index not in used_text:
            elements.append({
                "id": len(elements), "type": "text", "source": OCR_BACKEND,
                "description": item["text"], "ocr_text": item["text"],
                "detection_confidence": None, "ocr_confidence": item["ocr_confidence"],
                **coordinate_fields(item["box"], width, height),
            })
    parsed = {"image_sha256": digest, "width": width, "height": height,
              "elements": elements,
              "parser_settings": {"detection_threshold": detection_threshold,
                                  "ocr_threshold": ocr_threshold, "image_size": image_size,
                                  "caption_batch_size": batch_size, "ocr_backend": OCR_BACKEND}}
    PARSE_CACHE.clear()  # Keep only the latest screenshot to bound session memory.
    PARSE_CACHE[key] = parsed
    return image, parsed

print("Screenshot parser ready.")


## Step 4 · Target matching, optional fallback, and output

Florence descriptions and OCR labels are matched with whole words and a small synonym dictionary. This is transparent caption/label matching, not a universal reasoning model. Use short names; add synonyms in `TARGET_ALIASES` if a real failure needs one.

- `confidence`: an **uncalibrated heuristic ranking score**, not a probability of correctness.
- `detection_confidence`: OmniParser's confidence that a UI region exists; it does not identify the requested target.
- `ocr_confidence`: confidence in the recognized text, when available.
- `status`: `matched`, `ambiguous`, `not_found`, or `review_required`.

When similar candidates have close scores, the top-level box and center remain `null`; inspect the candidate list and narrow the region. The optional OpenCV fallback finds **right-facing triangular play shapes only**, is disabled by default, requires a search region, and always requests visual review.


In [ ]:
TARGET_ALIASES = {
    "play": ["play", "start playback", "resume playback", "play video", "right pointing triangle", "right facing triangle"],
    "pause": ["pause", "pause playback", "two vertical bars"],
    "like": ["like", "thumbs up", "heart", "favorite"],
    "dislike": ["dislike", "thumbs down"],
    "share": ["share", "sharing"],
    "comment": ["comment", "comments", "speech bubble", "chat bubble"],
    "volume": ["volume", "speaker", "sound"],
    "mute": ["mute", "muted", "speaker off"],
    "fullscreen": ["fullscreen", "full screen", "expand screen"],
    "close": ["close", "dismiss", "x icon", "cross icon"],
    "settings": ["settings", "gear", "cog"],
    "search": ["search", "magnifying glass", "magnifier"],
    "menu": ["menu", "hamburger", "three horizontal lines"],
    "bookmark": ["bookmark", "save"],
}

def words(text):
    return " ".join(re.findall(r"[\w]+", str(text).lower(), flags=re.UNICODE))

def contains_phrase(text, phrase):
    return f" {words(phrase)} " in f" {words(text)} "

def target_spec(target):
    normalized = words(target)
    meaningful = " ".join(token for token in normalized.split()
                          if token not in {"button", "icon", "control", "the", "a", "an"})
    if not meaningful:
        raise ValueError("Enter a target name, such as 'play button' or 'share icon'.")
    for canonical, aliases in TARGET_ALIASES.items():
        if meaningful == canonical or meaningful in [words(alias) for alias in aliases]:
            return canonical, aliases
    return meaningful, [meaningful]

def semantic_match(element, canonical, aliases):
    description = words(element.get("description", ""))
    # Prevent common neighboring actions from being mistaken for the requested action.
    blocked = {
        "play": ["pause", "replay", "autoplay", "playlist", "next", "previous", "skip"],
        "like": ["dislike", "thumbs down", "unlike", "unfavorite"],
        "volume": ["mute", "muted", "speaker off"],
        "fullscreen": ["exit fullscreen", "exit full screen", "minimize", "collapse"],
    }
    if any(contains_phrase(description, item) for item in blocked.get(canonical, [])):
        return 0.0
    if any(contains_phrase(description, alias) for alias in aliases):
        return 0.98 if description in [words(alias) for alias in aliases] else 0.92
    # For a target outside the dictionary, require all significant target words.
    tokens = set(canonical.split())
    return 0.80 if tokens and tokens <= set(description.split()) else 0.0

def validate_region(region):
    if region is None:
        return None
    if not isinstance(region, (list, tuple)) or len(region) != 4:
        raise ValueError("Search region must be [x1, y1, x2, y2] in normalized coordinates.")
    region = [float(v) for v in region]
    if not all(math.isfinite(v) and 0 <= v <= 1 for v in region):
        raise ValueError("Region coordinates must be finite numbers between 0 and 1.")
    if not (region[0] < region[2] and region[1] < region[3]):
        raise ValueError("The region must have positive width and height.")
    return region

def in_region(element, region, width, height):
    if region is None:
        return True
    x, y = element["center"][0] / width, element["center"][1] / height
    return region[0] <= x <= region[2] and region[1] <= y <= region[3]

def positional_bonus(element, preference, width, height):
    if preference == "any":
        return 0.0
    points = {"center": (0.5, 0.5), "bottom-left": (0.05, 0.95),
              "bottom-right": (0.95, 0.95), "top-left": (0.05, 0.05), "top-right": (0.95, 0.05)}
    if preference not in points:
        raise ValueError("Unknown position preference.")
    x, y = element["center"][0] / width, element["center"][1] / height
    px, py = points[preference]
    return 0.06 * max(0, 1 - math.hypot(x - px, y - py) / math.sqrt(2))

def rank_targets(elements, target, width, height, region=None, preference="any", min_score=0.70):
    if not 0 <= min_score <= 1:
        raise ValueError("Minimum match score must be between 0 and 1.")
    region = validate_region(region)
    canonical, aliases = target_spec(target)
    candidates = []
    for element in elements:
        if not in_region(element, region, width, height):
            continue
        match = semantic_match(element, canonical, aliases)
        if match == 0:
            continue
        evidence = element.get("detection_confidence")
        if evidence is None:
            evidence = element.get("ocr_confidence")
        evidence = float(evidence) if evidence is not None else 0.0
        score = min(1.0, 0.8 * match + 0.2 * evidence
                    + positional_bonus(element, preference, width, height))
        if score >= min_score:
            candidates.append({**element, "confidence": round(score, 6),
                               "semantic_match_score": match})
    candidates.sort(key=lambda item: (-item["confidence"], item["id"]))
    # Suppress OCR/region duplicates of the same control, preserving separate icons.
    unique = []
    for item in candidates:
        if not any(iou(item["box"], kept["box"]) >= 0.65 or
                   (intersection(item["box"], kept["box"]) / max(1, min(area(item["box"]), area(kept["box"]))) > 0.90
                    and max(area(item["box"]), area(kept["box"])) / max(1, min(area(item["box"]), area(kept["box"]))) < 3)
                   for kept in unique):
            unique.append(item)
    return unique

def opencv_play_shapes(image, region):
    """Geometric fallback only. Triangles in photographs can be false positives."""
    region = validate_region(region)
    if region is None:
        raise ValueError("Set SEARCH_REGION before enabling the OpenCV play fallback.")
    width, height = image.size
    left, top, right, bottom = clamp_box(
        [region[0] * width, region[1] * height, region[2] * width, region[3] * height], width, height)
    gray = cv2.cvtColor(np.asarray(image.crop((left, top, right, bottom))), cv2.COLOR_RGB2GRAY)
    output = []
    for mode in (cv2.THRESH_BINARY, cv2.THRESH_BINARY_INV):
        _, mask = cv2.threshold(gray, 0, 255, mode | cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            perimeter = cv2.arcLength(contour, True)
            polygon = cv2.approxPolyDP(contour, 0.04 * perimeter, True)
            if len(polygon) != 3 or not cv2.isContourConvex(polygon):
                continue
            x, y, w, h = cv2.boundingRect(polygon)
            if not (8 <= w <= min(160, width * 0.2) and 8 <= h <= min(160, height * 0.2)
                    and 0.4 <= w / h <= 1.6):
                continue
            points = polygon.reshape(-1, 2)
            tip_index = int(np.argmax(points[:, 0]))
            tip = points[tip_index]
            base = np.delete(points, tip_index, axis=0)
            if not (abs(float(base[0, 0]) - float(base[1, 0])) <= 0.25 * w
                    and tip[0] - base[:, 0].mean() >= 0.65 * w
                    and abs(tip[1] - base[:, 1].mean()) <= 0.22 * h):
                continue
            box = [left + x, top + y, left + x + w, top + y + h]
            if any(iou(box, item["box"]) > 0.5 for item in output):
                continue
            output.append({"id": 100000 + len(output), "type": "icon", "source": "opencv_triangle",
                           "description": "right facing triangle; unverified play shape", "ocr_text": "",
                           "detection_confidence": None, "ocr_confidence": None,
                           "confidence": 0.55, "semantic_match_score": None,
                           **coordinate_fields(box, width, height)})
    return output

def build_result(parsed, target, candidates, ambiguity_margin=0.08, fallback_used=False):
    if not 0 <= ambiguity_margin <= 1:
        raise ValueError("Ambiguity margin must be between 0 and 1.")
    ambiguous = len(candidates) > 1 and candidates[0]["confidence"] - candidates[1]["confidence"] < ambiguity_margin
    status = ("not_found" if not candidates else "review_required" if fallback_used
              else "ambiguous" if ambiguous else "matched")
    selected = candidates[0] if status == "matched" else None
    return {
        "target": target, "found": selected is not None, "status": status,
        "box": selected["box"] if selected else None,
        "center": selected["center"] if selected else None,
        "box_normalized": selected["box_normalized"] if selected else None,
        "confidence": selected["confidence"] if selected else None,
        "confidence_type": "uncalibrated_heuristic_not_probability",
        "detection_confidence": selected["detection_confidence"] if selected else None,
        "ocr_confidence": selected["ocr_confidence"] if selected else None,
        "selected_id": selected["id"] if selected else None,
        "ambiguous": ambiguous, "fallback_used": fallback_used,
        "image": {key: parsed[key] for key in ("width", "height", "image_sha256")},
        "coordinates": {"origin": "top_left", "units": "pixels", "box_format": "xyxy",
                        "right_bottom_edges": "exclusive", "reference": "full_exif_corrected_image"},
        "candidate_count": len(candidates), "candidates": candidates,
    }

def draw_boxes(image, elements, selected_id=None, region=None):
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    width, height = canvas.size
    line_width = max(2, round(max(width, height) / 700))
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", max(12, round(max(width, height) / 100)))
    except OSError:
        font = ImageFont.load_default()
    if region:
        draw.rectangle([region[0] * width, region[1] * height,
                        min(width - 1, region[2] * width), min(height - 1, region[3] * height)],
                       outline="#00bfff", width=line_width)
    for item in elements:
        color = "#00dd66" if item["id"] == selected_id else "#ffb000"
        x1, y1, x2, y2 = item["box"]
        draw.rectangle([x1, y1, x2 - 1, y2 - 1], outline=color, width=line_width)
        score = f" {item['confidence']:.2f}" if "confidence" in item else ""
        label = f"#{item['id']}{score}"
        text_box = draw.textbbox((0, 0), label, font=font)
        tw, th = text_box[2] - text_box[0], text_box[3] - text_box[1]
        tx, ty = max(0, min(x1, width - tw - 4)), max(0, y1 - th - 6)
        draw.rectangle((tx, ty, tx + tw + 4, ty + th + 5), fill=color)
        draw.text((tx + 2, ty - text_box[1] + 2), label, font=font, fill="black")
        if item["id"] == selected_id:
            cx, cy = item["center"]
            draw.ellipse((cx - 3, cy - 3, cx + 3, cy + 3), fill="red")
    return canvas

def detect_target(image_path, target, search_region=None, preference="any", min_score=0.70,
                  detection_threshold=0.10, ambiguity_margin=0.08, batch_size=8,
                  image_size=1280, enable_opencv_fallback=False):
    search_region = validate_region(search_region)
    canonical, _ = target_spec(target)
    if enable_opencv_fallback and search_region is None:
        raise ValueError("The optional triangle fallback needs a normalized SEARCH_REGION.")
    image, parsed = parse_screenshot(image_path, detection_threshold=detection_threshold,
                                     batch_size=batch_size, image_size=image_size)
    candidates = rank_targets(parsed["elements"], target, *image.size,
                              region=search_region, preference=preference, min_score=min_score)
    fallback_used = False
    if not candidates and enable_opencv_fallback and canonical == "play":
        candidates = opencv_play_shapes(image, search_region)
        fallback_used = bool(candidates)
    result = build_result(parsed, target, candidates, ambiguity_margin, fallback_used)
    result["search"] = {"region_normalized": search_region, "position_preference": preference,
                        "minimum_match_score": min_score, "ambiguity_margin": ambiguity_margin,
                        "opencv_fallback_enabled": enable_opencv_fallback}
    result["parser_settings"] = parsed["parser_settings"]
    result["models"] = {"repo": MODEL_REPO, "revision": MODEL_REVISION,
                         "florence_code_revision": FLORENCE_CODE_REVISION}
    result["created_at_utc"] = datetime.now(timezone.utc).isoformat()
    result["package_versions"] = {name: metadata.version(name) for name in
                                 ("torch", "transformers", "ultralytics", "numpy", OCR_BACKEND)}
    run_dir = WORK_DIR / "results" / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
                                      + "_" + parsed["image_sha256"][:8])
    run_dir.mkdir(parents=True, exist_ok=False)
    annotated = draw_boxes(image, candidates, result["selected_id"], search_region)
    annotated.save(run_dir / "target_annotated.png")
    draw_boxes(image, parsed["elements"]).save(run_dir / "all_elements.png")
    (run_dir / "result.json").write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    (run_dir / "all_elements.json").write_text(json.dumps(parsed, indent=2, ensure_ascii=False), encoding="utf-8")
    display(annotated)
    summary = {key: result[key] for key in ("target", "found", "status", "box", "center", "confidence",
                                           "confidence_type", "candidate_count")}
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    for candidate in candidates[:10]:
        print(f"#{candidate['id']} | score={candidate['confidence']:.3f} | {candidate['box']} | {candidate['description']}")
    if result["status"] == "not_found":
        print("No accepted match. Inspect all_elements.png/json; try another label or a screenshot with visible controls.")
    elif result["status"] == "ambiguous":
        print("Several plausible matches. Narrow SEARCH_REGION around the main player and rerun Step 6.")
    elif result["status"] == "review_required":
        print("OpenCV found possible play shapes. Inspect them visually; no target has been confirmed.")
    print("Saved:", run_dir)
    return result, run_dir

print("Target matching and export helpers ready.")


## Step 5 · Upload your app screenshot

Use an original-resolution PNG, JPEG, or WebP file. You can select several files; Step 6 lets you choose which one to analyze. Uploads remain files in your Colab session and are not sent to a vision-language API. This does not imply processing occurs on your own computer: the Colab runtime is remote.


In [ ]:
from google.colab import files

uploaded = files.upload()
SCREENSHOTS = []
for name, data in uploaded.items():
    safe_name = Path(name).name
    destination = WORK_DIR / "uploads" / (hashlib.sha256(data).hexdigest()[:12] + "_" + safe_name)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(data)
    try:
        preview = load_screenshot(destination)
    except (UnidentifiedImageError, OSError) as error:
        print(f"Skipped {safe_name}: unsupported or damaged image ({error}).")
        continue
    SCREENSHOTS.append(destination)
    print(f"[{len(SCREENSHOTS) - 1}] {safe_name} — {preview.width} × {preview.height} pixels")
    preview.thumbnail((1000, 650))
    display(preview)
if not SCREENSHOTS:
    raise RuntimeError("No valid screenshots uploaded. Rerun this cell and select an image file.")


## Step 6 · Enter the target and detect it

Start with `play button`. Other examples: `share icon`, `comment icon`, `like button`, `volume`, `fullscreen`, `close`, `search`, `settings`.

Leave **SEARCH_REGION** blank for the whole image. For multiple play controls, enter the main player's rectangle as normalized JSON, for example **`[0.0, 0.10, 0.75, 0.85]`**. This is an example only; use the boundaries of your screenshot. The blue rectangle shows the search region. It filters candidate centers without changing the coordinate origin.

**POSITION** adds a small ranking preference; it does not identify the main player automatically. Green indicates the selected match, amber shows other candidates. Set a region to resolve ambiguity. `CONFIDENCE_THRESHOLD` filters heuristic target scores; `DETECTION_THRESHOLD` filters proposed regions. Lowering either may increase false positives.


In [ ]:
# @title Find a UI target
TARGET = "play button" # @param {type:"string"}
SCREENSHOT_INDEX = 0 # @param {type:"integer"}
SEARCH_REGION = "" # @param {type:"string"}
POSITION = "any" # @param ["any", "center", "bottom-left", "bottom-right", "top-left", "top-right"]
CONFIDENCE_THRESHOLD = 0.70 # @param {type:"number"}
DETECTION_THRESHOLD = 0.10 # @param {type:"number"}
AMBIGUITY_MARGIN = 0.08 # @param {type:"number"}
CAPTION_BATCH_SIZE = 8 # @param {type:"integer"}
DETECTOR_IMAGE_SIZE = 1280 # @param {type:"integer"}
ENABLE_OPENCV_PLAY_FALLBACK = False # @param {type:"boolean"}

if "SCREENSHOTS" not in globals() or not SCREENSHOTS:
    raise RuntimeError("Upload an image in Step 5 first.")
if not 0 <= SCREENSHOT_INDEX < len(SCREENSHOTS):
    raise ValueError(f"SCREENSHOT_INDEX must be between 0 and {len(SCREENSHOTS) - 1}.")
try:
    region = json.loads(SEARCH_REGION) if SEARCH_REGION.strip() else None
except json.JSONDecodeError as error:
    raise ValueError("SEARCH_REGION must be blank or JSON such as [0.0, 0.1, 0.75, 0.85].") from error

LAST_RESULT, LAST_OUTPUT_DIR = detect_target(
    SCREENSHOTS[SCREENSHOT_INDEX], TARGET,
    search_region=region, preference=POSITION,
    min_score=CONFIDENCE_THRESHOLD, detection_threshold=DETECTION_THRESHOLD,
    ambiguity_margin=AMBIGUITY_MARGIN, batch_size=CAPTION_BATCH_SIZE,
    image_size=DETECTOR_IMAGE_SIZE, enable_opencv_fallback=ENABLE_OPENCV_PLAY_FALLBACK,
)


## Step 7 · Download the results

The ZIP contains `target_annotated.png`, `result.json`, `all_elements.png`, and `all_elements.json`. `result.json` contains pixel boxes, centers, normalized boxes, scores, status, and the complete accepted candidate list. `all_elements.json` contains every parsed element for debugging. Results are saved even when the target is not found.


In [ ]:
import shutil
from google.colab import files

if "LAST_OUTPUT_DIR" not in globals():
    raise RuntimeError("Run Step 6 first.")
archive = shutil.make_archive(str(LAST_OUTPUT_DIR), "zip", root_dir=LAST_OUTPUT_DIR)
files.download(archive)


## Troubleshooting and what changed

| Situation | Action |
|---|---|
| GPU memory error | Set `CAPTION_BATCH_SIZE = 2` or `1`; restart the session if necessary and run again. |
| Small icon missed | Try `DETECTOR_IMAGE_SIZE = 1920` and/or `DETECTION_THRESHOLD = 0.05`. Output still uses the original image coordinates. |
| More than one play button | Narrow `SEARCH_REGION` to the main player; inspect candidate IDs and descriptions. |
| Controls are hidden | Capture a new screenshot with controls visible. A static image cannot hover or reveal hidden UI. |
| Text or icon description is wrong | Inspect `all_elements.json`. Try a short synonym or switch OCR backend in Step 1 and restart. English OCR is configured by default; change the OCR language settings for other scripts. |
| Only a pause icon is visible | Search for `pause`; do not label it as a visible play icon. |
| OpenCV returns a triangle in the video | This is a shape heuristic. Keep fallback disabled, or narrow the region and visually review candidates. |
| Dependency/import error after install | Restart the Colab session and continue from Step 2. If you already installed other packages, start with a fresh runtime. |
| Python 3.13 / no compatible wheel | Select **Runtime Version 2026.07** (Python 3.12) with T4 GPU, then rerun Step 1. |

**Corrections from the uploaded version:** removed the invalid `from utils import ...` dependency; corrected `best.pt` to the actual `icon_detect/model.pt` weight; replaced the incorrect nested OCR unpacking and base64-to-array display assumptions with direct model/OCR outputs; avoided the upstream no-OCR merge edge case; preserved pixel `xyxy` boxes rather than confusing normalized `xywh` label-map values; added target entry, candidate ranking, ambiguity handling, explicit score meanings, optional play-shape fallback, and PNG/JSON downloads. Inference is cached for repeated target searches on the same screenshot.

**Verification:** notebook code and coordinate/matching/export logic were checked locally with controlled model/OCR outputs. Full pretrained-model inference still needs a GPU Colab run; no accuracy claim is made for YouTube, Netflix, or other apps until tested on real screenshots.

### Primary references
- [Microsoft OmniParser repository](https://github.com/microsoft/OmniParser)
- [OmniParser v2 weights and model card](https://huggingface.co/microsoft/OmniParser-v2.0)
- [OmniParser reference parser implementation](https://github.com/microsoft/OmniParser/blob/master/util/utils.py)
- [Florence-2 base-ft remote code](https://huggingface.co/microsoft/Florence-2-base-ft/tree/3afa95f29c831f1ea138d5d02e181fac2463e1e9)
- [Google Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html)
